# 05 · Waiter Effects — skill vs table luck (fixed effects)
Raw per-server averages confound *who* with *where they were assigned*. Two-way fixed effects (server + table + weekday) recover the causal-ish server lift — the statsmodels version of the production `adjustedGroupEffects` ridge.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from wineops_data import get_checks, get_tables, get_consumption, get_orders, get_inventory, daily_series
plt.rcParams['figure.figsize'] = (11, 4)

checks = get_checks()
checks['dow'] = pd.to_datetime(checks['opened_at'], format='ISO8601').dt.dayofweek
d = checks.dropna(subset=['server_name','table_id','total'])
raw = d.groupby('server_name')['total'].agg(['mean','size']).sort_values('mean', ascending=False)
raw.round(2)

In [ ]:
import statsmodels.formula.api as smf
fe = smf.ols('total ~ C(server_name) + C(table_id) + C(dow)', data=d).fit()
base = [s for s in sorted(d['server_name'].unique())][0]
eff = {base: 0.0}
for name, coef in fe.params.items():
    if name.startswith('C(server_name)'):
        eff[name.split('T.')[1].rstrip(']')] = coef
adj = pd.Series(eff).sort_values(ascending=False)
compare = pd.DataFrame({'raw_avg_check': raw['mean'], 'adjusted_effect_vs_' + base: adj}).round(2)
compare

In [ ]:
compare.plot.bar(subplots=True, layout=(1,2), figsize=(12,4), legend=False,
                 title=['Raw avg check (confounded)', 'Table+weekday-adjusted server effect'])
plt.tight_layout(); plt.show()

In [ ]:
# Wine attach rate per server, adjusted the same way (logit)
d2 = d.copy()
d2['items'] = d2['items'].apply(lambda x: x if isinstance(x, list) else [])
d2['has_wine'] = d2['items'].apply(lambda it: any(i.get('is_wine') for i in it)).astype(int)
logit = smf.logit('has_wine ~ C(server_name) + C(table_id) + C(dow)', data=d2).fit(disp=False)
att = {name.split('T.')[1].rstrip(']'): coef for name, coef in logit.params.items() if name.startswith('C(server_name)')}
pd.Series(att).sort_values(ascending=False).round(3)

**Read:** a server whose raw rank falls after adjustment was riding good tables; one who rises is genuinely lifting checks. Pair the top adjusted server with the bottom one for pre-shift pitch shadowing — then re-run in 3 weeks and measure whether the gap closed (that delta is the training ROI).